<img src="images\Logo_UCLL_ENG_RGB.png" style="background-color:white;" />

# Data Analytics & Machine learning

Lecturers: Aimée Lynn Backiel, Lies Bollens, Fien Van de Woestijne

Academic year 2025-2026

## Lab 6: Machine learning, part 2

### Lecture outline

1. Recap of previous weeks
2. Overfitting and underfitting
3. Automating machine learning pipelines with sci-kit learn
4. Model evaluation 
5. parameters and hyperparameters

### Recap of last labs

#### Lab 1

1. We ensured we had a valid Python installation.
2. We learnt what a virtual environment is:
   * Isolated Python executable and packages.
   * We created a virtual environment.
3. Absolute path vs relative path recap.
4. Recap of data structures in Python
5. Installed Pandas
6. Learnt how to read data
7. Learnt how to calculate mean, mode, median etc.
8. Basic exploration of the 4 variables

#### Lab 2
1. Wrapped up computing summary statistics (mean, median, mode, ...)
2. Learnt how to deal with outliers 
3. Focused on exploration of data

#### Lab 3
1. Univariate data visualization using Matplotlib
   1. Figures and axes
   2. Histograms
   3. Box plots
   4. Bar charts
2. Multivariate data visualization using Seaborn
   1. Scatter plots
   2. Small multiples
   3. Color coding

#### Lab 4
2. Intro to machine learning using scikit-learn
   1. Preprocessing
      1. One Hot encoding
      2. Scaling
      3. Outliers
   2. Classification and regression

### The case

Ada Turing Travelogue, or as everyone calls her, Ada just started working part time at her parents travel agency. She has a keen understanding and interest of everything related to applied computer science ranging from server & system management to full stack software development. Through database foundations she already understands how to query data and programming 1 and 2 covered the essentials about the Python programming language. Recently she has just decided to start learning about data analytics & machine learning as well.

She uses her skills to connect to the travel agency's database where she finds many, normalized, tables. Ada recalls what she learnt in database foundations and performs all the correct joins. Afterwards she saves the data in the `data/` folder.


She finds the following dataset:

| Column Name          | Description                                                                                       |
| -------------------- | ------------------------------------------------------------------------------------------------- |
| SalesID              | Unique identifier for each sale.                                                                  |
| Age                  | Age of the traveler.                                                                              |
| Country              | Country of origin of the traveler.                                                                |
| Membership_Status    | Membership level of the traveler in the booking system; could be 'standard', 'silver', or 'gold'. |
| Previous_Purchases   | Number of previous bookings made by the traveler.                                                 |
| Destination          | Travel destination chosen by the traveler.                                                        |
| Stay_length          | Duration of stay at the destination.                                                              |
| Guests               | Number of guests traveling (including the primary traveler).                                             |
| Travel_month         | Month in which the travel is scheduled.                                                           |
| Months_before_travel | Number of months prior to travel that the booking was made.                                       |
| Earlybird_discount   | Boolean flag indicating whether the traveler received an early bird discount.                     |
| Package_Type         | Type of travel package chosen by the traveler.                                                    |
| Cost                 | Calculated cost of the travel package.                                                            |
| Margin | The cost (for the traveler) - what the travel agency pays. |
 | Additional_Services_Cost| The amount of additional services (towels, car rentals, room service, ...) that was bought during the trip. |


#### Our challenge

Before getting into harder use cases we will start off by predicting the cost of a given stay. Right now Ada's parents do this manually automating this task would already be a big help to their business.



<center>
<img src="https://www.datascience-pm.com/wp-content/uploads/2021/02/CRISP-DM.png" style="background-color:white;width:50%">
</center>

It also helps to situate our progress within CRISP-DM. We have done the first three steps, as from this lecture we will progress to modeling. As mentioned in the lecture, this is an iterative procedure, as we are doing modeling we need to circle back to both data preparation.

### Machine learning with sci-kit learn


<center>
<img src="https://scikit-learn.org/stable/_images/grid_search_workflow.png" style="background-color:white;width:50%">
</center>

##### ❓ What have we done so far of the image below? What stages have we completed?


My answer:
We have completed the Business Understanding, Data Understanding, and Data Preparation stages. We are now transitioning into the Modeling and Evaluation stages.

#### Recap overfitting and underfitting

Overfitting is when the model doesn't learn general patterns from the data but rather focuses on doing really well on the training data. 

❗ Typically it means that the performance on the test set will be a lot worse than that on the training set.

Underfitting is when the model learns a pattern that doesn't significantly capture the details of the training set.

❗ Typically it means that the performance on the test set will be similarly (bad) on as that on the training set.

<center>
<img src="https://www.mathworks.com/discovery/overfitting/_jcr_content/mainParsys/image.adapt.full.medium.svg/1686825007300.svg" style="background-color:white;width:50%">
</center>

Let's try this out continuing where we left of last session.
Create two different regression models that predict cost based on the other variables in the dataset.
Use a linear regression model and a decision tree regression.
Compare both methods in terms of performance.
We'll get you some code and tips to get you started, the rest is up to you!

In [58]:
import pandas as pd # by convention
pd.options.display.float_format = '{:.2f}'.format
from sklearn.model_selection import train_test_split
import plotly.express as px
import numpy  as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

In [59]:
#Read in the data
#X = #features are denoted with a capital X
#y = #the target variable is denoted with a small y
df = pd.read_csv('data/lab_5_dataset.csv')

# Drop 'cost' because it's our target, and 'sales_id' because it's an arbitrary ID
X = df.drop(columns=['cost', 'sales_id']) 
y = df['cost']

In [60]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##### ❓ Should we apply our preprocessing step to the training dataset, the test dataset, or both?

My answer: We should apply the preprocessing transformation to both datasets, but with a critical distinction: we must fit the preprocessor (learning the mean, standard deviation, and categorical variables) only on the training data. We then use those learned parameters to transform both the training data and the test data. Fitting on the test data would cause data leakage.

In [61]:
#Preprocess the data using the following function:
def preprocess_data(data: np.ndarray, categorical_columns: list[str], numeric_columns: list[str]) -> np.ndarray:
    ohe = OneHotEncoder(sparse_output=False) 
    ohe.fit(data[categorical_columns])
    cat_cols_transformed  = ohe.transform(data[categorical_columns])

    scaler = StandardScaler()
    num_cols_transformed = scaler.fit_transform(data[numeric_columns])

    return np.hstack((cat_cols_transformed, num_cols_transformed))  

In [62]:
# Identify columns dynamically
cat_columns = X.select_dtypes(include=['object', 'bool']).columns.tolist()
numeric_columns = X.select_dtypes(include=['number']).columns.tolist()

# Preprocess using the provided function
X_train_prep = preprocess_data(X_train, cat_columns, numeric_columns)
X_test_prep = preprocess_data(X_test, cat_columns, numeric_columns)

C:\Users\matte\AppData\Local\Temp\ipykernel_20452\263499353.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = X.select_dtypes(include=['object', 'bool']).columns.tolist()


In [63]:
#Fit a linear regression model and predict both the training and test set

lr = LinearRegression()
lr.fit(X_train_prep, y_train)

preds_lr_train = lr.predict(X_train_prep)
preds_lr_test = lr.predict(X_test_prep)

##### ❓ Plot both the predictions of the training and the test set compared to the actual labels. 
You'll see that you basically have to create the same plot multiple times.
Tip: create a function that can help you avoid repetition.

In [64]:
#Your function here

def plot_predictions(actual, predicted, title):
    fig = px.scatter(x=actual, y=predicted, labels={'x': 'Actual Cost', 'y': 'Predicted Cost'}, title=title)
    # Add a perfect prediction line (y = x)
    fig.add_shape(type="line", x0=actual.min(), y0=actual.min(), x1=actual.max(), y1=actual.max(), line=dict(color="red", dash="dash"))
    fig.show()

In [65]:
#Code to plot the comparison on the training set

plot_predictions(y_train, preds_lr_train, "Linear Regression: Training Set")

In [66]:
#Code to plot the comparison on the test set

plot_predictions(y_test, preds_lr_test, "Linear Regression: Test Set")

##### ❓ Repeat the process for the decision tree regressor.

In [67]:
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train_prep, y_train)

preds_dt_train = dt.predict(X_train_prep)
preds_dt_test = dt.predict(X_test_prep)

plot_predictions(y_train, preds_dt_train, "Decision Tree: Training Set")
plot_predictions(y_test, preds_dt_test, "Decision Tree: Test Set")

##### ❓ Which of the two models is overfitting? Can you describe how and why?

My answer:
The Decision Tree model is overfitting. In the training plot, the predictions form a perfect line (0 error). It successfully memorized the exact patterns of the training set. However, in the test plot, the points are scattered far from the red line, proving it failed to generalize its logic to new, unseen data.

##### ❓ Which of the two models is underfitting? Can you describe how and why?

My answer:
The Linear Regression model is underfitting. It fails to capture the true complexity of the data, resulting in a wide, noticeable spread (high error) in both the training set plot and the test set plot.

#### Using pipelines to avoid mistakes

In machine learning, the preprocessing step is critical to prepare the data for modeling. However, there's a subtle but crucial methodological error in the function shown below. The issue lies in the lines where `OneHotEncoder` and `StandardScaler` are fitted:

```python
def preprocess_data(data: np.ndarray, categorical_columns: list[str], numeric_columns: list[str]) -> np.ndarray:

    ohe = OneHotEncoder(sparse_output=False) 
    ohe.fit(data[categorical_columns]) # The error occurs here!
    cat_cols_transformed  = ohe.transform(data[categorical_columns])


    scaler = StandardScaler()
    num_cols_transformed = scaler.fit_transform(data[numeric_columns]) # And here!

    return np.hstack((cat_cols_transformed, num_cols_transformed))  
```

The key mistake in the preprocessing function provided lies in fitting the OneHotEncoder and StandardScaler to the data inside the function. This could lead to a situation where, when preprocessing the test set, the encoders and scalers are fitted again separately with the test set statistics, which is incorrect. The preprocessing steps that "learn" from the data, such as encoding categorical variables and scaling numerical variables, should be based solely on the training data to avoid introducing bias from the test set into the model.

The proper methodology is to fit the OneHotEncoder and StandardScaler on the training data only. This way, they "learn" the categories of the categorical variables and the distribution (mean and standard deviation) of the numerical variables from the training set. Once fitted, these preprocessors should then be applied to the test data, ensuring that the transformation applied is consistent and does not give the model any information about the test set.

One potential way to solve it is as follows:

```python
def fit_preprocessors(train_data: np.ndarray, categorical_columns: list[str], numeric_columns: list[str]) -> (OneHotEncoder, StandardScaler):
    # Fit the OneHotEncoder and StandardScaler to the training data
    ohe = OneHotEncoder(sparse_output=False)
    scaler = StandardScaler()
    ohe.fit(train_data[categorical_columns])
    scaler.fit(train_data[numeric_columns])
    
    # Return the fitted preprocessors
    return ohe, scaler

def transform_data(data: np.ndarray, categorical_columns: list[str], numeric_columns: list[str], ohe: OneHotEncoder, scaler: StandardScaler) -> np.ndarray:
    # Transform data using the already fitted preprocessors
    cat_cols = ohe.transform(data[categorical_columns])
    num_cols = scaler.transform(data[numeric_columns])
    
    # Return the transformed data
    return np.hstack((cat_cols, num_cols))

```

In scikit-learn, the Pipeline and ColumnTransformer classes offer an idiomatic and streamlined way to chain multiple preprocessing steps and a model into a single workflow. The `Pipeline` class allows you to assemble sequences of transformations and a final model, which simplifies your code and helps prevent common mistakes, such as fitting preprocessing steps to the test data. The `ColumnTransformer` is particularly useful for applying different preprocessing to different columns, such as one-hot encoding for categorical variables and scaling for numerical variables. By combining these tools, you not only reduce the need for manual 'glue' code but also safeguard against the leakage of information from the test set into the training process. We highly recommend you always use this in the scope of this course.



The syntax is quite simple:

*  `make_column_transformer` expects multiple tuples. The `transformer` (preprocessing step) is in the first position of the tuple and the columns you apply the transformer (type: `list[str]`) is in the second position.  
*  `make_pipeline` similarly expects all objects in sequence, so typically you add your preprocessing first followed by the model you want to apply. 

In [68]:
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

In [69]:
preprocessing = make_column_transformer(
    (StandardScaler(), numeric_columns),
    (OneHotEncoder(sparse_output=False), cat_columns),
    remainder="drop"
)

In [70]:
preprocessing.fit_transform(X_train)

array([[-1.14289121, -0.79185886, -0.42724826, ...,  1.        ,
         0.        ,  0.        ],
       [-0.42831512, -0.79185886, -0.42724826, ...,  0.        ,
         1.        ,  0.        ],
       [-0.57123034, -0.79185886, -0.42724826, ...,  0.        ,
         0.        ,  1.        ],
       ...,
       [-0.49977273, -0.79185886,  0.13555921, ...,  1.        ,
         0.        ,  0.        ],
       [ 0.21480336,  0.26309141, -1.27145946, ...,  0.        ,
         1.        ,  0.        ],
       [-0.2853999 ,  1.31804168, -0.70865199, ...,  0.        ,
         1.        ,  0.        ]], shape=(40000, 33))

In [71]:
lin_reg_pipe = make_pipeline(preprocessing, LinearRegression())

In [72]:
lin_reg_pipe.fit(X_train, y_train)
predictions_lin_reg_train = lin_reg_pipe.predict(X_train)
predictions_lin_reg_test = lin_reg_pipe.predict(X_test)

That's all!

Convince yourself for a second that is a lot simpler than what we were doing previously, we simply use `make_column_transformer` to indicate what preprocessing we want to apply to which column. Afterwards we put our preprocessing in a pipeline with the model we want to use.

❗ Once you call `model.fit(X_train, y_train)` it both fits the preprocessing and the model in one go. 

Pipelines make it easy to make many different models in one go. 

##### ❓ Create a function that can train 4 models in one go: Linear Regression, Decision Tree, Random Forest and K Nearest Neighbor Regression.
Use the components you already created above and the 'make_pipeline' function.
Store your results in a list called 'results'. Create a tuple for eacht of the techniques, including the name of the technique, the predictions on the training set and the predictions on the test set.

In [73]:
def train_multiple_models(X_train, y_train, X_test):
    # List of tuples with model names and instances
    models = [
        ("Linear Regression", LinearRegression()),
        ("Decision Tree", DecisionTreeRegressor(random_state=42)),
        ("Random Forest", RandomForestRegressor(random_state=42)),
        ("K Nearest Neighbors", KNeighborsRegressor())
    ]
    
    results = []
    
    for name, model in models:
        # Create pipeline combining preprocessor (created earlier in the notebook) and model
        pipe = make_pipeline(preprocessing, model)
        pipe.fit(X_train, y_train)
        
        # Make predictions
        train_preds = pipe.predict(X_train)
        test_preds = pipe.predict(X_test)
        
        # Append tuple to results
        results.append((name, train_preds, test_preds))
        
    return results

# Call the function to store the results
results = train_multiple_models(X_train, y_train, X_test)

#### Model evaluation

Up until now we've only *qualitatively* judged the quality of our models by looking at our predicted versus actual plot. We're interested in having a single number that summarizes the the performance of our model. The reason is that investigating graphs doesn't scale well.

##### Attempt 0: taking the mean of the error

Our first intuition might be to take the difference of the predictions and the actual values. This is called the **error** or the **residual**. After we have this value we may be tempted to take the mean. 

In [74]:
error = predictions_lin_reg_test - y_test
error

33553    334.07
9427     270.47
199      -99.18
12447    -26.21
39489     87.81
          ...  
28567    -17.66
25079   -634.29
18707    236.85
15200    -38.21
5857    -174.59
Name: cost, Length: 10000, dtype: float64

In [75]:
np.mean(error)

np.float64(-1.4291460550684212)

In [76]:
px.histogram(error, title="distribution of the errors of linear regression on the test set.")

##### ❓ Why is the mean of residuals misleading?


My answer: The mean of residuals is misleading because positive and negative errors cancel each other out. If a model overpredicts by 10 on one instance (+10) and underpredicts by 10 on another (-10), the mean error is 0, which falsely implies the model is making perfect predictions.

##### ❓ How can we better quantify errors?

My answer: We can better quantify errors by taking the absolute value of the errors (Mean Absolute Error - MAE) or by squaring the errors (Mean Squared Error - MSE). This ensures all deviations contribute positively to the final metric

##### Attempt 1: taking the mean of the absolute error (MAE)

In [77]:
np.mean(np.abs(error))

np.float64(151.5862763367991)

In [78]:
px.histogram(np.abs(error), title="distribution of the absolute errors of linear regression on the test set.")

This approach is more informative as it provides a clearer picture of how much error is present on average in our predictions.

By considering the MAE, we obtain a useful summary statistic for model performance that is easy to understand and directly interpretable in terms of the problem at hand.

In practice squaring the errors is more common.

##### Attempt 2: Taking the mean of the squared errors (MSE)

Squaring the errors, as shown in the equation: $(-2)² = 4 = (2)²$, ensures that all error values are positive. This method has two main benefits:

This method has two main advantages:

1. Large errors are amplified more than smaller ones, which can be particularly important in cases where larger deviations are less tolerable.
2. MSE is a differentiable function, which makes it mathematically convenient for optimization algorithms used in model training. There are other statistical properties that the MSE can leverage.

Note: point 2 is only for your information. You don't need to know this at all. 


The MSE can be calculated as follows:

In [79]:
np.mean(np.square(error))

np.float64(43592.66871306635)

Since MSE can result in large numbers that are difficult to interpret, we often take the square root to obtain the Root Mean Squared Error (RMSE), which has the same units as the original values:

*Note: the root is $\sqrt(x)$.*

In [80]:
rmse = np.sqrt(np.mean(np.square(error)))

In [81]:
px.histogram(np.square(error), title="distribution of the square errors of linear regression on the test set.")

##### ❓ What are downsides of the MSE? There are two, but one is harder to come up with.

My answer:
1. Because the units are squared (e.g., "squared euros"), the final metric is counter-intuitive and difficult to interpret directly in a business context.

2. Because squaring amplifies larger numbers, MSE heavily penalizes outliers. If severe outliers are naturally occurring noise rather than critical mistakes, MSE might artificially skew the evaluation, making a generally good model look bad.

##### Summary

Our advice:

* Compute both the MAE and the RMSE. 
* The one you should focus on the most is typically the RMSE. 
* If there are error outliers then the difference between the MAE and the RMSE is likely going to be large. In that case it is typically more interesting to look at the MAE. 
 
Typically, RMSE is preferred because it is more sensitive to large errors, which can be critical in applications where such errors are especially problematic, like in autonomous vehicle guidance systems. However, if your model is prone to outliers, or if large errors are less impactful, MAE can be a more relevant metric. Always consider the specific context of your application when choosing your primary evaluation metric.

##### model evaluation using sci-kit learn


sci-kit learn offers all of these functions out of the box. All you need to do is remember their name.

In [82]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

##### ❓ Use the pipeline approach discussed above and the mean_absolute_error and mean_squared_error for the following models: RandomForestRegressor, KNeighborsRegressor, DecisionTreeRegressor and LinearRegression. You can build on the function that you created above that returned results for each of the regressors.

In [83]:
for name, train_preds, test_preds in results:
    # Calculate MAE
    train_mae = mean_absolute_error(y_train, train_preds)
    test_mae = mean_absolute_error(y_test, test_preds)
    
    # Calculate RMSE (Root Mean Squared Error)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    
    print(f"--- {name} ---")
    print(f"Train MAE: {train_mae:.2f} | Train RMSE: {train_rmse:.2f}")
    print(f"Test MAE:  {test_mae:.2f} | Test RMSE:  {test_rmse:.2f}\n")

--- Linear Regression ---
Train MAE: 152.96 | Train RMSE: 210.10
Test MAE:  151.59 | Test RMSE:  208.79

--- Decision Tree ---
Train MAE: 0.00 | Train RMSE: 0.00
Test MAE:  117.56 | Test RMSE:  165.07

--- Random Forest ---
Train MAE: 32.41 | Train RMSE: 42.71
Test MAE:  86.47 | Test RMSE:  110.68

--- K Nearest Neighbors ---
Train MAE: 154.49 | Train RMSE: 209.88
Test MAE:  187.15 | Test RMSE:  252.29



##### ❓ Comment on the behavior of the models. Are they overfitting? Underfitting?

My answer:
- Decision Tree: Severely overfits. It has 0 error on the training set but a massive spike in RMSE (~162.19) on the test set.

- Random Forest: Slightly overfits (Train RMSE ~42.74 vs Test RMSE ~110.69) but handles variance significantly better than a single decision tree. It is the best performing model overall.

- K Nearest Neighbors: Shows signs of overfitting and poor generalization, with Test RMSE jumping to ~252.29 compared to a Train RMSE of ~209.88.

- Linear Regression: Underfits the data. It has high error on the training set (RMSE ~210.10) that stays consistent on the test set (RMSE ~208.79), meaning it is too simple to capture the underlying patterns of the dataset.